# Session 2: Validation, Exceptions, and Debugging

**Time:** 60-90 minutes  
**Method:** Predict -> Run -> Investigate -> Modify -> Make -> Retrieve later

You write every TODO. The examples teach the syntax, but they solve different problems. Run setup and check cells as provided.

## 0. Retrieval warm-up

Answer from memory before looking at Session 1.

In [ ]:
warmup = {
    "return_vs_print": "print only displays while return sends the data back to caller",
    "meaning_of_arrow_none": "-> None means a function returns nothing/ no data",
    "do_type_hints_enforce_types": "No they only shows the expected datatype",
}

## 1. Preconditions and validation

A **precondition** is something that must be true before a function can do its job correctly. Validation checks those conditions near the function boundary. This produces a clear error instead of allowing bad data to cause a confusing failure later.

Example contract: a training epoch count must be a positive integer.

In [3]:
def validate_epochs(epochs: int) -> None:
    """Raise ValueError when the epoch count is not positive."""
    if epochs <= 0:
        raise ValueError("epochs must be greater than zero")


valid_result = validate_epochs(10)
print(valid_result)

None


### Syntax breakdown

- `if epochs <= 0:` checks the invalid condition.
- `raise` stops normal execution and reports a failure.
- `ValueError(...)` means the value is unacceptable for this function.
- `-> None` means successful validation returns no useful value. Silence means the value passed.

Common mistake: returning `False` without checking it at the call site. Raising an exception cannot be silently mistaken for success.

## 2. Expected validation failures

An exception caused by deliberately invalid input is expected behavior, not necessarily a programming bug. `try` runs risky code. `except ValueError as error` handles only the expected exception type.

In [4]:
try:
    validate_epochs(0)
except ValueError as error:
    print(type(error).__name__)
    print(error)

ValueError
epochs must be greater than zero


Do not use `except Exception:` by default. It can hide unrelated bugs such as misspelled names or incorrect operations. Catch the specific failure you expect.

## 3. Modify: validate a probability

A probability must be between `0.0` and `1.0`, including both boundaries.

**Function contract**

- Parameter: `probability`, expected type `float`.
- Successful return: `None`.
- Failure: raise `ValueError` for values below 0 or above 1.

Syntax hint: combine two invalid comparisons with `or`.

In [5]:
def validate_probability(probability: float) -> None:
    """Raise ValueError when probability is outside the valid range."""
    # TODO: check the invalid range and raise ValueError with a useful message.
    if probability > 1 or probability < 0:
        raise ValueError("Probability should be in range of 0 and 1")

<details><summary>Stronger hint</summary>Write one `if` condition containing `probability < 0` and `probability > 1`. Join them with `or`.</details>

In [6]:
# CHECK: run after completing validate_probability.
assert validate_probability(0.0) is None
assert validate_probability(0.5) is None
assert validate_probability(1.0) is None

for invalid_probability in (-0.01, 1.01):
    try:
        validate_probability(invalid_probability)
        raise AssertionError("Expected ValueError for an invalid probability")
    except ValueError:
        pass

print("Passed: probability boundaries and invalid values")

Passed: probability boundaries and invalid values


## 4. Reading a traceback

Read a traceback from the final line upward:

1. Final line: exception type and message.
2. Line immediately above: operation that failed.
3. Earlier frames: calls that led there.

The next example captures the traceback so the notebook can continue running.

In [7]:
import traceback


def percentage_to_fraction(percentage: float) -> float:
    return percentage / 100


try:
    percentage_to_fraction("eighty")
except TypeError:
    captured_traceback = traceback.format_exc()
    print(captured_traceback)

Traceback (most recent call last):
  File "C:\Users\Pony\AppData\Local\Temp\ipykernel_22736\481173682.py", line 9, in <module>
    percentage_to_fraction("eighty")
  File "C:\Users\Pony\AppData\Local\Temp\ipykernel_22736\481173682.py", line 5, in percentage_to_fraction
    return percentage / 100
           ~~~~~~~~~~~^~~~~
TypeError: unsupported operand type(s) for /: 'str' and 'int'



In [ ]:
traceback_analysis = {
    "exception_type": "TypeError",
    "failed_operation": "/ on string",
    "smallest_reproduction": "percentage_to_fraction('eighty')",
    "testable_hypothesis": "the percentage argument passed by the caller was not compatible ",
}

## 5. Investigate and fix broken validation

The function below intends to accept only binary labels, `0` and `1`, but its condition is wrong.

First predict which inputs incorrectly pass or fail. Then modify only the condition.

Syntax hint: membership is written as `value not in (0, 1)`.

In [8]:
broken_validator_prediction = "all values greater than 0 even 0.2, 2.2 and all"


def validate_binary_label(label: int) -> None:
    """Raise ValueError when label is not 0 or 1."""
    # TODO: this condition is wrong. Replace it.
    if label not in (0,1):
        raise ValueError("label must be 0 or 1")

In [9]:
# CHECK: run after fixing validate_binary_label.
assert validate_binary_label(0) is None
assert validate_binary_label(1) is None

for invalid_label in (-1, 2, 7):
    try:
        validate_binary_label(invalid_label)
        raise AssertionError("Expected ValueError for a non-binary label")
    except ValueError:
        pass

print("Passed: binary-label validation")

Passed: binary-label validation


## 6. Make: validated accuracy

Rebuild `calculate_accuracy` with explicit preconditions. Do not copy a completed solution.

Required behavior:

- `y_true` and `y_pred` must have equal lengths.
- They must not be empty.
- Every value in both lists must be `0` or `1`.
- Valid input returns `matching labels / number of labels` as `float`.

Useful syntax:

```python
if condition:
    raise ValueError("useful message")

for label in labels:
    validate_binary_label(label)
```

Parameters are the two lists. The return value is one float. `ValueError` is part of the function contract.

In [ ]:
def calculate_accuracy(y_true: list[int], y_pred: list[int]) -> float:
    """Return binary classification accuracy for two validated label lists.

    Args:
        y_true: Correct binary labels.
        y_pred: Predicted binary labels.

    Returns:
        The fraction of positions containing matching labels.

    Raises:
        ValueError: If lengths differ, lists are empty, or labels are non-binary.
    """
    # TODO: validate all preconditions, then calculate and return accuracy.

    if len(y_true) == 0 or len(y_pred) == 0:
        raise ValueError("Length of the lists should not be zero")
    if len(y_true) != len(y_pred):
        raise ValueError("Length should be equal for the y_true and y_pred lists")

    for label1 in y_true:
        validate_binary_label(label1)

    for label1 in y_pred:
        validate_binary_label(label1)

    count = 0
    true_count = len(y_true)
    for i in range(len(y_true)):
        if y_true[i] == y_pred[i]:
            count+=1 
    return count / true_count
    


<details><summary>Stronger hint</summary>Perform structural checks first: unequal lengths, then empty input. Next loop through both lists and call `validate_binary_label`. Calculate matches only after validation succeeds.</details>

In [23]:
# SETUP: run this helper. Understanding every line is not required yet.
def assert_raises_value_error(function, *args) -> None:
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("Expected ValueError, but no ValueError was raised")

In [24]:
# AUTOMATED CHECKS
assert calculate_accuracy([1, 0, 1], [1, 0, 1]) == 1.0
assert calculate_accuracy([1, 0, 1, 0], [1, 1, 1, 1]) == 0.5
assert_raises_value_error(calculate_accuracy, [1, 0], [1])
assert_raises_value_error(calculate_accuracy, [], [])
assert_raises_value_error(calculate_accuracy, [1, 2], [1, 0])
assert_raises_value_error(calculate_accuracy, [1, 0], [1, -1])
print("Passed: valid, mismatched, empty, and invalid-label cases")

Passed: valid, mismatched, empty, and invalid-label cases


## 7. Explain your decisions

Use full sentences. `Good` or copied lesson text does not count.

In [ ]:
explanation = {
    "what_is_a_precondition": "A condition which should be true before a function starts working on data",
    "why_validation_returns_none": "that means all checks passed",
    "why_raise_instead_of_return_false": "because we need to know what error occured and debug and fix that",
    "expected_failure_vs_bug": "expected failures are just bad entries in data while bug is a flawed logic",
    "traceback_reading_order": "starts from the very last which shows the name of the error and then above you see which line caused it",
    "why_validate_before_calculating": "we should validate as it will help us debug and erases confusion",
}

most_difficult_part = "What raise returns"
help_used = "None"
one_question_i_still_have = "You can tell me in chat that what raise actually do and does that abort the function entirely? as when we were checking labels"

## Retrieval task: complete 2-3 days later

Without reopening this lesson, write from memory:

1. A validator returning `None` on success and raising `ValueError` on failure.
2. The order used to read a traceback.
3. One smallest reproducible example for a failure you encountered.

In [ ]:
retrieval_date = ""
retrieval_validator = ""
retrieval_traceback_order = ""
retrieval_smallest_reproduction = ""

## Save and stop

Save the notebook after all immediate checks pass. Do not complete the retrieval task today. Send the notebook for review before committing.